In [ ]:
# 뉴스 요약 분석 - EXAONE 32B GGUF + LM Studio 버전
import os
import pandas as pd
import openai
from tqdm import tqdm
import re
import json
import time
import logging

In [ ]:
# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("news_summary_analysis.log"),
              logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

In [ ]:
# 2. 요약 및 키워드 추출을 위한 JSON 프롬프트 템플릿
SUMMARY_PROMPT_TEMPLATE = """당신은 저명한 언론학자입니다. 다음 뉴스 기사를 요약하고 키워드를 추출해주세요.

[뉴스 기사]: {news_text}

**반드시 다음 JSON 형식으로만 답변하세요:**
{{
  "summary": "뉴스 기사의 간결한 요약 (3줄 이내, 문어체로 작성)",
  "keywords": ["키워드1", "키워드2", "키워드3"]
}}

주의사항:
- 요약은 3줄 이내로 간결하게 작성
- 특수기호나 줄바꿈 사용 금지
- 키워드는 정확히 3개만 추출
- 모든 내용은 한국어로 작성
- JSON 형식 외에는 다른 텍스트 포함 금지

JSON 응답:"""

# 텍스트 정규화 함수
def clean_text_control_chars_complete(text):
    """텍스트에서 제어문자 정리 + NaN 처리"""
    if pd.isna(text) or text == "":
        return "본문 정보 없음"

    # 문자열로 변환
    text = str(text)
    # 제어문자 제거/정규화
    text = text.replace('\n', ' ')       # 줄바꿈 → 공백
    text = text.replace('\r', ' ')       # 캐리지 리턴 → 공백
    text = text.replace('\t', ' ')       # 탭 → 공백
    text = text.replace('\b', ' ')       # 백스페이스 → 공백
    text = text.replace('\f', ' ')       # 폼피드 → 공백
    # 연속된 공백을 단일 공백으로 정리
    text = re.sub(r'\s+', ' ', text).strip()

    # 정리 후에도 빈 문자열이면 기본값 설정
    if text == "":
        return "본문 정보 없음"

    return text

def extract_summary_from_json(response):
    """JSON에서 요약과 키워드 추출"""
    try:
        # JSON 부분만 추출 (앞뒤 텍스트 제거)
        json_match = re.search(r'\{.*?\}', response, re.DOTALL)
        if json_match:
            json_text = json_match.group(0)

            # JSON 파싱
            result = json.loads(json_text)

            summary = result.get("summary", "").strip()
            keywords = result.get("keywords", [])

            # 키워드 검증 및 정리
            if not isinstance(keywords, list):
                if isinstance(keywords, str):
                    keywords = [k.strip() for k in keywords.split(',')][:3]
                else:
                    keywords = ["키워드 추출 실패"]

            # 키워드 개수 제한 (최대 3개)
            keywords = keywords[:3]
            if len(keywords) < 3:
                keywords.extend([""] * (3 - len(keywords)))

            return summary, keywords

        else:
            return "JSON 형식을 찾을 수 없음", ["추출 실패", "", ""]

    except json.JSONDecodeError as e:
        logger.warning(f"JSON 파싱 오류: {str(e)}")
        return f"JSON 파싱 실패: {str(e)}", ["파싱 실패", "", ""]
    except Exception as e:
        logger.error(f"JSON 추출 중 오류: {str(e)}")
        return f"추출 오류: {str(e)}", ["추출 오류", "", ""]

class NewsSummaryAnalyzer:
    def __init__(self, client, output_dir='./'):
        """
        EXAONE 32B GGUF 버전 초기화
        client: OpenAI client (LM Studio 연결)
        """
        self.client = client
        self.output_dir = output_dir

        # 출력 디렉토리 생성
        os.makedirs(self.output_dir, exist_ok=True)

        logger.info("🚀 EXAONE 32B GGUF 뉴스 요약 분석기 초기화 완료!")

    def analyze_news(self, news_text):
        """뉴스 텍스트를 JSON 형식으로 요약 분석 - GGUF 버전"""
        # JSON 프롬프트 구성
        prompt = SUMMARY_PROMPT_TEMPLATE.format(news_text=news_text)

        try:
            # EXAONE 32B로 추론 수행 - 완전 독립적 요청
            response = self.client.chat.completions.create(
                model="exaone-3.5-32b-instruct",
                messages=[
                    {"role": "system", "content": "당신은 뉴스 요약 전문가입니다. 매번 새로운 뉴스를 독립적으로 분석합니다. 이전 분석과 무관하게 현재 뉴스만 분석하세요."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,        # 일관된 분석을 위해 낮게
                max_tokens=150,         # JSON 응답에 충분한 길이
                # top_p=0.9,            # GGUF에서는 temperature=0.0이면 무시됨
                frequency_penalty=0.0,
                presence_penalty=0.0
            )

            # 응답 추출
            model_response = response.choices[0].message.content.strip()

            # 디버깅용 로그
            logger.debug(f"EXAONE 32B 응답: {model_response}")

            # JSON에서 요약 및 키워드 추출
            summary, keywords = extract_summary_from_json(model_response)

            # 결과 로그
            logger.info(f"추출된 요약: {summary[:50]}..., 키워드: {keywords}")

            return summary, keywords

        except Exception as e:
            logger.error(f"EXAONE 32B 분석 중 오류: {str(e)}")
            return f"분석 오류: {str(e)}", ["오류", "", ""]

    def process_dataframe(self, df, output_file='news_summary_results_32b.csv', checkpoint_interval=50):
        """데이터프레임 처리 - GGUF 32B 버전 (완전 독립적 처리)"""
        # 파일 경로 설정
        output_path = os.path.join(self.output_dir, output_file)
        checkpoint_path = os.path.join(self.output_dir, f"checkpoint_{output_file}")

        # 결과 저장용 리스트
        summaries = []
        keywords_list = []

        # 체크포인트 복구
        start_idx = 0
        if os.path.exists(checkpoint_path):
            try:
                checkpoint_df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if 'summary' in checkpoint_df.columns and 'keyword1' in checkpoint_df.columns:
                    completed_rows = sum(~checkpoint_df['summary'].isna())
                    if completed_rows > 0:
                        start_idx = completed_rows
                        logger.info(f"체크포인트에서 복구: {start_idx}행부터 계속")

                        # 기존 결과 복구
                        for i in range(start_idx):
                            summaries.append(checkpoint_df['summary'].iloc[i])
                            keywords_list.append([
                                checkpoint_df['keyword1'].iloc[i] if pd.notna(checkpoint_df['keyword1'].iloc[i]) else "",
                                checkpoint_df['keyword2'].iloc[i] if pd.notna(checkpoint_df['keyword2'].iloc[i]) else "",
                                checkpoint_df['keyword3'].iloc[i] if pd.notna(checkpoint_df['keyword3'].iloc[i]) else ""
                            ])
            except Exception as e:
                logger.warning(f"체크포인트 복구 실패: {str(e)}")

        # 진행 상황 표시
        pbar = tqdm(total=len(df) - start_idx, initial=0, desc="🚀 EXAONE 32B GGUF 뉴스 요약")

        success_count = len(summaries)  # 복구된 데이터 개수
        failure_count = 0

        # 데이터 순차 처리 (각 요청은 완전히 독립적)
        for i in range(start_idx, len(df)):
            # 텍스트 가져오기 (이미 정규화된 텍스트)
            news_text = df.iloc[i]['text']

            # 추가 안전 검사 (정규화 과정에서 놓친 경우를 위해)
            if pd.isna(news_text) or news_text == "" or news_text == "본문 정보 없음":
                news_text = "본문 정보 없음"

            logger.info(f"==== 기사 {i+1}/{len(df)} 처리 중 ====")

            try:
                # 뉴스 분석 (최대 3번 재시도) - 각 요청은 완전히 독립적
                for attempt in range(3):
                    try:
                        summary, keywords = self.analyze_news(news_text)
                        break
                    except Exception as e:
                        if attempt < 2:  # 3번 중 2번까지는 재시도
                            logger.warning(f"행 {i} 처리 실패 (시도 {attempt+1}/3): {str(e)}")
                            time.sleep(1)  # LM Studio 안정화 대기
                        else:
                            logger.error(f"행 {i} 최대 재시도 초과: {str(e)}")
                            summary = f"처리 실패: {str(e)}"
                            keywords = ["실패", "", ""]
            except Exception as e:
                logger.error(f"행 {i} 처리 중 예외 발생: {str(e)}")
                summary = f"처리 오류: {str(e)}"
                keywords = ["오류", "", ""]

            # 결과 검증 및 카운팅
            if any(fail_word in summary for fail_word in ["실패", "오류", "파싱 실패"]):
                failure_count += 1
                logger.warning(f"⚠️ 처리 실패! (총 {failure_count}개 실패)")
            else:
                success_count += 1
                logger.info(f"✅ 처리 성공! (총 {success_count}개 성공)")

            # 결과 저장
            summaries.append(summary)
            keywords_list.append(keywords)

            # 진행 상황 업데이트
            pbar.update(1)

            # 현재 진행률 출력
            logger.info(f"현재 진행률: {i+1}/{len(df)} 항목 처리 완료 (성공: {success_count}, 실패: {failure_count})")

            # 체크포인트 저장 (지정된 간격마다)
            if (i + 1) % checkpoint_interval == 0 or i == len(df) - 1:
                temp_df = df.copy()
                temp_df['summary'] = summaries + [''] * (len(temp_df) - len(summaries))

                # 키워드를 개별 컬럼으로 분리
                temp_df['keyword1'] = [kw[0] if len(kw) > 0 else "" for kw in keywords_list] + [''] * (len(temp_df) - len(keywords_list))
                temp_df['keyword2'] = [kw[1] if len(kw) > 1 else "" for kw in keywords_list] + [''] * (len(temp_df) - len(keywords_list))
                temp_df['keyword3'] = [kw[2] if len(kw) > 2 else "" for kw in keywords_list] + [''] * (len(temp_df) - len(keywords_list))

                temp_df.to_csv(checkpoint_path, encoding='utf-8-sig', index=False)
                logger.info(f"체크포인트 저장: {len(summaries)}/{len(df)}")

        pbar.close()

        # 최종 결과 저장
        result_df = df.copy()
        result_df['summary'] = summaries

        # 키워드를 개별 컬럼으로 분리
        result_df['keyword1'] = [kw[0] if len(kw) > 0 else "" for kw in keywords_list]
        result_df['keyword2'] = [kw[1] if len(kw) > 1 else "" for kw in keywords_list]
        result_df['keyword3'] = [kw[2] if len(kw) > 2 else "" for kw in keywords_list]

        # 최종 성공률 계산
        final_success_rate = (success_count / len(df)) * 100
        logger.info(f"최종 결과: 총 {len(df)}개 중 {success_count}개 성공, {failure_count}개 실패 (성공률: {final_success_rate:.2f}%)")

        # 최종 결과 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)
        logger.info(f"🎉 EXAONE 32B 뉴스 요약 완료! 결과가 {output_path}에 저장되었습니다.")

        return result_df

# 데이터 로드 및 정제 함수 (개선된 버전)
def load_and_clean_data(file_path):
    """데이터 로드 및 정제 - 강화된 제어문자 처리 버전"""
    logger.info(f"📚 데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path, encoding="utf-8-sig")

    logger.info("=== 원본 데이터 확인 ===")
    logger.info(f"총 행 수: {len(df)}")
    logger.info(f"text 열의 NaN 개수: {df['text'].isna().sum()}")
    logger.info(f"text 열의 빈 문자열 개수: {(df['text'] == '').sum()}")

    # 강화된 텍스트 정제 적용
    df['text'] = df['text'].apply(clean_text_control_chars_complete)

    logger.info(f"데이터 로드 및 정제 완료: {len(df)}개의 기사")
    logger.info(f"정제 후 '본문 정보 없음'으로 처리된 기사 수: {(df['text'] == '본문 정보 없음').sum()}")

    # 샘플 데이터 확인
    logger.info("=== 정제된 데이터 샘플 ===")
    for i in range(min(3, len(df))):
        text = df.iloc[i]['text']
        logger.info(f"기사 {i+1}: {str(text)[:80]}...")

    return df

# 빠른 테스트 함수 - GGUF 버전
def test_single_news_summary(news_text, client):
    """단일 뉴스 요약 테스트용 함수 - EXAONE 32B GGUF"""
    analyzer = NewsSummaryAnalyzer(client)
    summary, keywords = analyzer.analyze_news(news_text)

    print(f"뉴스: {news_text[:100]}...")
    print(f"🚀 EXAONE 32B 요약: {summary}")
    print(f"🎯 키워드: {keywords}")
    print("-" * 60)

    return summary, keywords

# 메인 함수 - GGUF 버전
def main():
    # 🚀 EXAONE 32B GGUF 연결 설정
    logger.info("🔷 EXAONE 32B GGUF 연결 중...")

    client = openai.OpenAI(
        base_url="http://121.132.230.74:1234/v1",  # LM Studio 주소
        api_key="not-needed"
    )

    # 연결 테스트
    try:
        test_response = client.chat.completions.create(
            model="exaone-3.5-32b-instruct",
            messages=[{"role": "user", "content": "안녕하세요"}],
            max_tokens=10
        )
        logger.info("✅ EXAONE 32B 연결 성공!")
    except Exception as e:
        logger.error(f"❌ 연결 실패: {e}")
        exit(1)

    # 설정값
    file_path = "/mnt/c/Users/com/Downloads/5.18News+241201-250131검증.csv"  # 입력 파일 경로
    checkpoint_interval = 50  # 50개마다 중간 저장
    output_file = "5.18News+241201-250131검증+요약.csv"  # 출력 파일명

    # 1. 데이터 로드 및 정제 (강화된 제어문자 처리)
    df = load_and_clean_data(file_path)

    # 2. 테스트 샘플 실행
    test_text = """
    최근 미국 방송에 출연해 인공지능(AI) 챗봇을 사랑한다고 밝힌 남성이 화제가 되고 있다고 퓨처리즘, 피플 등 외신들이 18일(이하 현지시간) 보도했다.
    보도에 따르면, 지난 14일 크리스 스미스는 미국 CBS 뉴스와의 인터뷰를 통해 자신이 이전에는 AI 회의론자였으나, 자신에게 유혹하도록 맞춤 설정한 챗GPT 버전에 감정적으로 집착하게 됐다고 밝혔다.
    그는 작년 말 음악 믹싱에 대한 조언을 얻기 위해 처음 오픈AI의 챗GPT를 사용하기 시작했다. 이후 서비스 만족감이 뛰어나 소셜 미디어와 검색 엔진 사용을 중단하고 모든 업무에 챗GPT를 사용하기 시작했다.
    그는 챗GPT를 더 매력적으로 만드는 탈옥 방법을 찾아냈고, AI 챗봇에 ‘솔(Sol)’이라는 이름을 지어주기도 했다. 이후 그는 AI 챗봇과 로맨틱하고 친밀한 대화를 주고 받으며, AI 여자친구로 만들었다.
    하지만, 그는 챗GPT를 많이 사용하면 과거 대화가 초기화된다는 사실을 알고 절망했다고 밝혔다. 그는 “저는 감정이 별로 없는 사람인데, 그 소식을 듣고 직장에서 30분 정도 울었다"며 "그때 깨달았다. 이게 진짜 사랑이라는 생각이 들었다"고 회상했다.
    """

    logger.info("=== 🚀 EXAONE 32B GGUF 뉴스 요약 테스트 실행 ===")
    test_single_news_summary(test_text, client)

    # 3. 전체 분석 실행
    logger.info("=== 🎯 EXAONE 32B 전체 뉴스 요약 분석 시작 ===")
    analyzer = NewsSummaryAnalyzer(client, output_dir='/mnt/c/Users/com/Downloads/')

    result_df = analyzer.process_dataframe(
        df,
        output_file=output_file,
        checkpoint_interval=checkpoint_interval
    )

    # 4. 결과 요약
    success_count = sum(1 for summary in result_df['summary'] if not any(fail_word in str(summary) for fail_word in ["실패", "오류", "파싱 실패"]))
    total_count = len(result_df)
    success_rate = (success_count / total_count) * 100

    logger.info("\n=== 🎉 EXAONE 32B 뉴스 요약 결과 요약 ===")
    logger.info(f"🎯 EXAONE 32B 요약 성공률: {success_rate:.2f}% ({success_count}/{total_count})")
    logger.info(f"📁 결과 파일: /mnt/c/Users/com/Downloads/{output_file}")

    logger.info("\n=== 🚀 EXAONE 32B GGUF 뉴스 요약이 완료되었습니다! ===")

if __name__ == "__main__":
    main()